# 📊 Estatística Básica — Pedro A. Morettin & Wilton de O. Bussab

Este notebook implementa em Python os exemplos e exercícios do **Capítulo 02 - Resumo de Dados** do livro Estatística Básica 10ª Edição (Pedro A. Morettin & Wilton O. Bussab).

Concebido como material de apoio ao livro-texto, o código inclui adaptações pontuais na estruturação dos dados e funções autorais para otimizar a análise e visualização.

## 1. Configuração do Ambiente e Registro de Metadados

Gerenciamento de versões e rastreabilidade da execução utilizando a extensão `watermark`.

In [1]:
# Instalação das bibliotecas necessárias no ambiente Jupyter
%pip install -q -U watermark
%pip install pyprojroot
%pip install xlrd

print('✅ Instalações concluídas! Verifique a necessidade de reiniciar o kernel.')

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✅ Instalações concluídas! Verifique a necessidade de reiniciar o kernel.


In [2]:
# Ativação do watermark e registro de autoria
%load_ext autoreload
%autoreload
%reload_ext watermark
%watermark -a 'Rafael F. Santos'

Author: Rafael F. Santos



In [ ]:
# Importação das bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xlrd
import sys
from pyprojroot import here
from src.stats_utils import criar_tabela_frequencias
import warnings

# Adiciona a raiz automaticamente, não importa a profundidade da pasta
sys.path.insert(0, str(here()))

# Ignora alertas
warnings.filterwarnings('ignore')

print('✅ Importações executadas com sucesso!')

✅ Importações executadas com sucesso!


# Função para criar uma tabela de frequências
import pandas as pd


def criar_tabela_frequencias(dados: pd.Series, ordem_categorias=None) -> pd.DataFrame:
    """Gera uma tabela de distribuição de frequências para uma série de dados.

    Calcula a frequência absoluta, a frequência relativa (proporção) e a
    porcentagem de cada categoria, adicionando uma linha final com os totais.

    Parâmetros:
    -----------
    dados : pd.Series
        Série contendo os dados brutos a serem contabilizados.
    ordem_categorias : list, tuple ou pd.Index, opcional
        Sequência definindo a ordem desejada para as categorias no índice.
        Se None, mantém a ordenação decrescente padrão por contagem.

    Retorno:
    --------
    pd.DataFrame
        Tabela com as colunas:
        - 'Frequência (ni)': Contagem absoluta de ocorrências.
        - 'Proporção (fi)': Frequência relativa (entre 0 e 1), arredondada para 2 casas decimais.
        - 'Porcentagem (100 * fi)': Frequência percentual (entre 0% e 100%).
        Linha 'Total' com a soma de cada coluna.
    """
    # Contagem de ocorrências absolutas
    freq_absoluta = dados.value_counts()

    # Cálculo da proporção e porcentagem reaproveitando a contagem
    freq_relativa = (freq_absoluta / freq_absoluta.sum()).round(4)
    freq_percentual = (freq_relativa * 100).round(2)

    # Construção do DataFrame com nomes de colunas padronizados
    tabela = pd.DataFrame({
        'Frequência (ni)': freq_absoluta,
        'Proporção (fi)': freq_relativa,
        'Porcentagem (100 * fi)': freq_percentual
    })

    # Aplicação de ordenação personalizada, se fornecida
    if ordem_categorias is not None:
        tabela = tabela.reindex(ordem_categorias)
    else:
        tabela = tabela.sort_index()

    # Linha de totalização calculada diretamente sobre as colunas da tabela
    tabela.loc['Total'] = [
        tabela['Frequência (ni)'].sum(),
        tabela['Proporção (fi)'].sum().round(),
        tabela['Porcentagem (100 * fi)'].sum().round()
    ]

    return tabela

## 2. Importando dados

O primeiro passo para a análise de dados é a importação dos dados.

In [5]:
# Importar os dados da tabela
tab21 = pd.read_csv('../data/raw/tabela2_1.csv', sep=';')

# Visualizar amostra
tab21.head()

,N,estado_civil,grau_instrucao,n_filhos,salario,idade_anos,idade_meses,reg_procedencia
0,1,solteiro,ensino fundamental,NaN,"4,00",26,3,interior
1,2,casado,ensino fundamental,1.0,"4,56",32,10,capital
2,3,casado,ensino fundamental,2.0,"5,25",36,5,capital
3,4,solteiro,ensino médio,NaN,"5,73",20,10,outra
4,5,solteiro,ensino fundamental,NaN,"6,26",40,7,outra


In [6]:
# Verificar as variáveis da tabela
tab21.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   N                36 non-null     int64  
 1   estado_civil     36 non-null     str    
 2   grau_instrucao   36 non-null     str    
 3   n_filhos         20 non-null     float64
 4   salario          36 non-null     str    
 5   idade_anos       36 non-null     int64  
 6   idade_meses      36 non-null     int64  
 7   reg_procedencia  36 non-null     str    
dtypes: float64(1), int64(3), str(4)
memory usage: 3.5 KB


In [7]:
# Verificar as variáveis da tabela
tab21.columns

Index(['N', 'estado_civil', 'grau_instrucao', 'n_filhos', 'salario',
       'idade_anos', 'idade_meses', 'reg_procedencia'],
      dtype='str')

In [8]:
# Criar a tabela de frequências de 'grau_instrucao'
ordem = ['ensino fundamental', 'ensino médio', 'superior']
dados = tab21['grau_instrucao']

# Tabela de Frequências
criar_tabela_frequencias(dados, ordem)

,Frequência (ni),Proporção (fi),Porcentagem (100 * fi)
grau_instrucao,,,
ensino fundamental,12.0,0.3333,33.33
ensino médio,18.0,0.5000,50.00
superior,6.0,0.1667,16.67
Total,36.0,1.0000,100.00


## Problemas

[1.] **Escalas de medidas**. A seguir descrevemos outros possíveis critérios para classificar variáveis, em função da escala adotada. Observe a similaridade com a classificação apresentada anteriormente. Nossas observações são resultados de medidas 
feitas sobre os elementos de uma população. Existem quatro escalas de medidas que podem ser consideradas:

**Escala nominal**. Nesta escala, somente podemos afirmar que uma medida, é diferente ou não de outra, e ela é usada para categorizar indivíduos de uma população. Um exemplo é o sexo de um indivíduo. Para cada categoria associamos um numeral 
diferente (letra ou número). Por exemplo, no caso do sexo, podemos associar as letras M (masculino) e F (feminino) ou 1 (masculino) e 2 (feminino). Não podemos realizar operações aritméticas aqui e uma medida de posição apropriada é a moda. (As 
medidas citadas nesse problema, como a média, mediana e moda, são definidas no Capítulo 3.)

**Escala ordinal**. Aqui, podemos dizer que uma medida é diferente e maior do que outra. Temos a situação anterior, mas as categorias são ordenadas, e a ordem dos numerais associados ordena as categorias. Por exemplo, a classe socioeconômica de um 
indivíduo pode ser baixa(1 ou X), média (2 ou Y) e alta (3 ou Z). Transformações que preservam a ordem não alteram a estrutura de uma escala ordinal. No exemplo acima, podemos representar as categorias por 1, 10 e 100 ou A, L e Z. Medidas de posição 
apropriadas são a mediana e a moda.

**Escala intervalar**. Nesta escala, podemos afirmar que uma medida é igual ou diferente, maior e quanto maior do que outra. Podemos quantificar a diferença entre as categorias da escala ordinal. Necessitamos de uma origem arbitrária e de uma 
unidade de medida. Por exemplo, considere a temperatura de um indivíduo, na escala Fahrenheit. A origem é $0º F$ e a unidade é $1º F$, Transformações que preservam a estrutura dessa escala são do tipo $y= ax + b, a > O$. Por exemplo, a transformação 
$y = 5/9 (x - 32)$ transforma graus Fahrenheit em centígrados. Para essa escala, podemos fazer operações aritméticas, assim média, mediana e moda são medidas de posição apropriadas.

**Escala razão**. Dadas duas medidas nessa escala, podemos dizer se são iguais, ou se uma é diferente, maior, quanto maior e quantas vezes a outra. A diferença com a escala intervalar é que agora existe um zero absoluto. A altura de um indivíduo é 
um exemplo de medida nessa escala. Se ela for medida em centímetros (cm), O cm é a origem e 1 cm é a unidade de medida. Um indivíduo com 190 cm é duas vezes mais alto do que um indivíduo com 95 cm, e esta relação continua a valer se usarmos 1 m como 
unidade. Ou seja, a estrutura da escala razão não é alterada por transformações da forma $y = cx, c > O$. Por exemplo, $y = x/100$ transforma cm em m. As estatísticas apropriadas para a escala intervalar são também apropriadas para a escala razão.

Para cada uma das variáveis abaixo, indique a escala usualmente adotada para resumir os dados em tabelas de frequências:

**(a)** Salários dos empregados de uma indústria.  
**Resposta:** Escala razão   

**(b)** Opinião de consumidores sobre determinado produto.  
**Resposta:** Escala ordinal   

**(c)** Número de respostas certas de alunos num teste com dez itens.  
**Resposta:** Escala razão  

**(d)** Temperatura diária da cidade de Manaus.  
**Resposta:** Escala intervalar   

**(e)** Porcentagem da receita de municípios aplicada em educação.  
**Resposta:** Escala razão  

**(f)** Opinião dos empregados da Companhia MB sobre a realização ou não de cursos obrigatórios de treinamento.  
**Resposta:** Escala nominal  

**(g)** Qi de um indivíduo.  
**Resposta:** Escala intervalar

[2.] Usando os dados da Tabela 2.1, construa a distribuição de frequências das variáveis:

(a) Estado civil.  
(b) Região de procedência.  
(c) Número de filhos dos empregados casados.  
(d) Idade.

In [ ]:
# Tabela de frequências do Estado Civil
ordem = ['casado', 'solteiro']
dados = tab21['estado_civil']

criar_tabela_frequencias(dados, ordem)

In [ ]:
# Tabela de frequências da Região de procedência
# ordem = ['capital', 'interior', 'outra']
dados = tab21['reg_procedencia']

criar_tabela_frequencias(dados)


In [ ]:
# Tabela de frequências do Número de filhos dos empregados casados
dados = tab21[tab21['estado_civil'] == 'casado'].groupby(by='estado_civil')['n_filhos']

criar_tabela_frequencias(dados)

In [ ]:
# Tabela de frequências da Idade
dados = tab21['idade_anos']

criar_tabela_frequencias(dados)

[3.] Para o Conjunto de Dados 1 (CD-Brasil), construa a distribuição de frequências para as variáveis população e densidade populacional.

In [ ]:
import numpy as np
import pandas as pd

# 1. Ler o arquivo pulando as 5 primeiras linhas de título/fonte
df = pd.read_excel("../data/raw/Dados_EB.xls", sheet_name='CD-Brasil', skiprows=5, header=None)

# 2. Definir e renomear as colunas corretamente
df.columns = ["Regiao", "UF", "Superficie", "Populacao", "Densidade"]

# 3. Remover a primeira linha se ela contiver os nomes antigos dos cabeçalhos textuais
# (Geralmente fica como a linha index 0 após pular o skiprows)
df = df[df["UF"] != "UF"].reset_index(drop=True)

# 4. Preencher as células vazias da Região (Propagação para baixo)
# Como a região só aparece na primeira linha, o ffill() repete ela nas linhas de baixo vazias
df["Regiao"] = df["Regiao"].ffill()

# 5. Filtrar e remover as linhas de Subtotais e Totais do meio dos dados
df = df[~df["UF"].str.contains("Subtot|Total", na=False, case=False)]

# 6. Garantir que as colunas numéricas não tenham textos e sejam lidas como números limpos
# Caso o Excel tenha lido como texto devido a pontos/vírgulas, isso resolve:
# for col in ["Superficie_km2", "Populacao", "Densidade_hab_km2"]:
#     if df[col].dtype == "object":
#         df[col] = (
#             df[col]
#             .astype(str)
#             .str.replace(".", "", regex=False)
#             .str.replace(",", ".", regex=False)
#         )
#     df[col] = pd.to_numeric(df[col], errors="coerce")

# 7. Resetar o índice final para ficar limpo (0, 1, 2...)
df.reset_index(drop=True, inplace=True)

# Visualizar o resultado final perfeito
df.head()

In [ ]:
df.info()

## 5. Relatório do Ambiente de Execução

In [ ]:
# Detalhes de autoria e versões das bibliotecas carregadas
%watermark -a 'Rafael F. Santos' --iversions

In [ ]:
# Versão do interpretador Python e especificações do sistema
%watermark -m -v

**Fim do Notebook.**